In [6]:
# Prev code - copy pasted

import numpy as np
import math

batch_size = 2 #Two sentences at once
seq_len = 3
d_model = 8
num_heads = 2
d_k = d_model // num_heads # 4

# Our input X (batch_size, seq_len, d_model) -> (2, 3, 8)
X = np.random.randn(batch_size, seq_len, d_model)

# The massive weight matrices (d_model, d_model) -> (8, 8)
# Notice they project d_model to d_model, not d_model to d_k!
W_Q = np.random.randn(d_model, d_model)
W_K = np.random.randn(d_model, d_model)
W_V = np.random.randn(d_model, d_model)
W_O = np.random.randn(d_model, d_model) # The final output projection (Section 3.2)


### PART 1 ###
# Project X to get Q, K, V

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Reshape to include num_heads
Q_reshaped = Q.reshape(batch_size, seq_len, num_heads, d_k)
K_reshaped = K.reshape(batch_size, seq_len, num_heads, d_k)
V_reshaped = V.reshape(batch_size, seq_len, num_heads, d_k)

# Transpose to get the num_heads to a before axis
Q_reshaped_t = Q_reshaped.transpose(0,2,1,3)
K_reshaped_t = K_reshaped.transpose(0,2,1,3)
V_reshaped_t = V_reshaped.transpose(0,2,1,3)

# Calculating score
S = Q_reshaped_t @ K_reshaped_t.transpose(0,1,3,2)
S = S / (math.sqrt(d_k))

# Applying Softmax

S_max_sub = S - np.max(S, axis=-1, keepdims=True)
S_max_sub_epow = np.exp(S_max_sub)
S_sum = np.sum(S_max_sub_epow, axis=-1, keepdims=True)
S_normalised = S_max_sub_epow/S_sum

# Calculate output

output = S_normalised @ V_reshaped_t

# Reshape output

output_t = output.transpose(0,2,1,3)
output_reshaped = output_t.reshape(batch_size, seq_len, d_model)

# Final Y

Y = output_reshaped @ W_O

### PART 2 ###
# Applying residual connections

X_residual = X + Y
print(f"Shape of X_residual is {X_residual.shape}")

# Layer Norm
epsilon = 1e-5
gamma = np.ones(d_model)
beta = np.zeros(d_model)

X_residual_mean = np.mean(X_residual, axis=-1, keepdims=True)
X_residual_var = np.var(X_residual, axis=-1, keepdims=True)
X_centered = (X_residual - X_residual_mean) / np.sqrt(X_residual_var + epsilon)
X_norm = gamma * X_centered + beta

### PART 3 ###
# Constants 
d_ff = d_model * 4

W_1 = np.random.randn(d_model, d_ff)
W_2 = np.random.randn(d_ff, d_model)


# First linear transformation. Assume bias to be zero
H_1 = X_norm @ W_1
print(f"Shape of H_1 is {H_1.shape}")

# Second transformation is non-linear. Let's use RELU max(0,x)
H_2 = np.maximum(H_1, 0)
# Note : np.max and np.maximum are different functions. Check documentation


# Third transformation is linear again. This will reduce size back to d_model
H_3 = H_2 @ W_2
print(f"Shape of H_3 is {H_3.shape}")


### PART 4 ### 

# Applying second residual and layer norm

X_residual_2 = X_norm + H_3
X_residual_2_mean = np.mean(X_residual_2, axis=-1, keepdims=True)
X_residual_2_var = np.var(X_residual_2, axis=-1, keepdims=True)
X_centered_2 = (X_residual_2 - X_residual_2_mean) / np.sqrt(X_residual_2_var + epsilon)
Y_final = gamma * X_centered_2 + beta

print(f"Shape of Y_final is {Y_final.shape}")

Shape of X_residual is (2, 3, 8)
Shape of H_1 is (2, 3, 32)
Shape of H_3 is (2, 3, 8)
Shape of Y_final is (2, 3, 8)
